# 02 — Feature Engineering
Build and persist user, item, and session features ready for model training.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import pandas as pd
from utils.config import load_config

cfg = load_config()
PROJECT_ROOT = os.path.dirname(os.getcwd())
print("Config loaded. Project root:", PROJECT_ROOT)

## 1. Load Raw Data

In [ ]:
events        = pd.read_csv(f"{PROJECT_ROOT}/data/raw/events.csv")
category_tree = pd.read_csv(f"{PROJECT_ROOT}/data/raw/category_tree.csv")
item_props    = pd.concat([
    pd.read_csv(f"{PROJECT_ROOT}/data/raw/item_properties_part1.csv"),
    pd.read_csv(f"{PROJECT_ROOT}/data/raw/item_properties_part2.csv"),
], ignore_index=True)

print(f"Events         : {len(events):>10,}")
print(f"Item properties: {len(item_props):>10,}")
print(f"Category tree  : {len(category_tree):>10,}")

## 2. User Features & Interaction Matrix

In [ ]:
from features.user_features import build_user_features, build_user_item_matrix

user_features   = build_user_features(events)
interactions    = build_user_item_matrix(events)

print(f"User features shape   : {user_features.shape}")
print(f"Cold-start users      : {user_features['is_cold_start'].mean():.1%}")
print(f"Interaction matrix    : {len(interactions):,} rows")
user_features.head(3)

## 3. Item Features & TF-IDF Matrix

In [ ]:
from features.item_features import build_item_features, build_item_tfidf_matrix

item_features               = build_item_features(item_props, category_tree)
item_ids, tfidf_mat, vec    = build_item_tfidf_matrix(item_props, max_features=5000)

print(f"Item features shape : {item_features.shape}")
print(f"TF-IDF matrix shape : {tfidf_mat.shape}")
item_features.head(3)

## 4. Session Segmentation & Sequences

In [ ]:
from features.session_features import build_sessions, build_session_sequences, build_session_features

sessions_df   = build_sessions(events, session_gap_hours=1.0)
sequences     = build_session_sequences(sessions_df, max_len=20, min_len=2)
session_feats = build_session_features(sessions_df)

print(f"Total sessions        : {sessions_df['session_id'].nunique():,}")
print(f"Training sequences    : {len(sequences):,}")
print(f"Session features shape: {session_feats.shape}")
sequences.head(3)

## 5. Persist Processed Features

In [ ]:
import os, scipy.sparse as sp, pickle

PROCESSED = f"{PROJECT_ROOT}/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

user_features.to_parquet(f"{PROCESSED}/user_features.parquet", index=False)
item_features.to_parquet(f"{PROCESSED}/item_features.parquet", index=False)
interactions.to_parquet(f"{PROCESSED}/interactions.parquet", index=False)
sequences.to_parquet(f"{PROCESSED}/session_sequences.parquet", index=False)
session_feats.to_parquet(f"{PROCESSED}/session_features.parquet", index=False)
sp.save_npz(f"{PROCESSED}/tfidf_matrix.npz", tfidf_mat)
item_ids.to_csv(f"{PROCESSED}/tfidf_item_ids.csv", index=False)
with open(f"{PROCESSED}/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vec, f)

print("All processed features saved to", PROCESSED)